<a href="https://colab.research.google.com/github/ChariteshReddyPatlolla/stock-price-predector/blob/main/stock_price_liveupdate.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import yfinance as yf
import pandas as pd
from sklearn.linear_model import SGDRegressor
from sklearn.preprocessing import StandardScaler
import numpy as np
import time

# Initialize
model = SGDRegressor()
scaler = StandardScaler()

X=[]
y=[]

# Initial training (last 5 days, 1-minute data)
def initial_train():
    data = yf.download("AAPL", period="5d", interval="1m")
    data = data.reset_index()
    data["timestamp"] = data.index
    data["minutes"] = np.arange(len(data))
    X = data[["minutes"]].values
    y = data["Close"].values

    X_scaled = scaler.fit_transform(X)
    model.partial_fit(X_scaled, y)

# Live update + retraining
def live_update():
    i = len(y)  # start from end of initial data
    while True:
        new_data = yf.download("AAPL", period="1m", interval="1m")
        if not new_data.empty:
            price = new_data["Close"][-1]
            now = len(y) + i
            X_new = scaler.transform([[now]])
            prediction = model.predict(X_new)[0]
            print(f"[{now}] Predicted: {prediction:.2f}, Actual: {price:.2f}")

            # Update model with new data point
            model.partial_fit(X_new, [price])
            i += 1
        time.sleep(60)

initial_train()
live_update()


/tmp/ipython-input-2-2161195123.py:17: FutureWarning: YF.download() has changed argument auto_adjust default to True
  data = yf.download("AAPL", period="5d", interval="1m")
[*********************100%***********************]  1 of 1 completed
/usr/local/lib/python3.11/dist-packages/sklearn/utils/validation.py:1408: DataConversionWarning: A column-vector y was passed when a 1d array was expected. Please change the shape of y to (n_samples, ), for example using ravel().
  y = column_or_1d(y, warn=True)
/tmp/ipython-input-2-2161195123.py:31: FutureWarning: YF.download() has changed argument auto_adjust default to True
  new_data = yf.download("AAPL", period="1m", interval="1m")
[*********************100%***********************]  1 of 1 completed
ERROR:yfinance:
1 Failed download:
ERROR:yfinance:['AAPL']: YFInvalidPeriodError("AAPL: Period '1m' is invalid, must be one of: 1d, 5d, 1mo, 3mo, 6mo, 1y, 2y, 5y, 10y, ytd, max")
/tmp/ipython-input-2-2161195123.py:31: FutureWarning: YF.download() 

KeyboardInterrupt: 